# Lab Day 10 — Nguyễn Thị Thùy Dương

Qwen3-VL-4B: E0–E4 + bonus E5. Mã nguồn được đóng gói trong notebook; công cụ đề giữ nguyên.
Chọn **T4 GPU**, chạy các cell lần lượt. Kết quả chỉ xuất hiện sau khi huấn luyện/suy luận thật.


## 0. Môi trường (Colab T4)
Giải nén source và cài đúng versions của đề.


In [ ]:
import base64, io, os, sys, zipfile, subprocess
from pathlib import Path
ROOT = Path('/content/lab_day10')
ROOT.mkdir(exist_ok=True)
SOURCE_B64 = (
    '...'
    'ACwAWgsAAKQKAQAAAA=='
)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(SOURCE_B64))) as z:
    z.extractall(ROOT)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
# Put this notebook in ROOT too, so the instructor's notebook-structure test can run.
from solution.make_student_notebook import build
build()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-student.txt'], check=True)
from solution.colab_environment import prepare
prepare()
import torch
assert torch.cuda.is_available(), 'Runtime → Change runtime type → T4 GPU'
print(torch.cuda.get_device_name(0), torch.cuda.get_device_properties(0).total_memory / 2**30, 'GiB')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
WORK = Path('/content/drive/MyDrive/Day10_ThuyDuong/checkpoints')
WORK.mkdir(parents=True, exist_ok=True)
ARTIFACTS = WORK.parent / 'report'
ARTIFACTS.mkdir(exist_ok=True)
if not Path('report').exists():
    Path('report').symlink_to(ARTIFACTS, target_is_directory=True)
print('Checkpoint:', WORK, '\nEvidence:', ARTIFACTS)
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests', 'student_tests'], check=True)


## 1. Dữ liệu: tải và chia split
Dataset công khai tải qua Kagglehub; split do công cụ gốc của đề tạo.


In [ ]:
import kagglehub
raw = kagglehub.dataset_download('lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes')
subprocess.run([sys.executable, 'scripts/prepare_data.py', '--source', raw], check=True)
from solution.experiment import validate_data
print(validate_data())


## 2. Nhãn: YOLO → JSON thang 0–1000
Kiểm tra nhãn một ảnh và vẽ box để đối chiếu.


In [ ]:
from scripts.labcommon import read_jsonl
from solution.core import objects_to_answer, prompt
from solution.reporting import annotate
from IPython.display import display
from PIL import Image
train = read_jsonl('data/splits/train.jsonl')
for i, entry in enumerate(train[:6]):
    answer = objects_to_answer(entry['objects'])
    annotate(entry, answer, 'data', f'report/figures/ground_truth_{i}.png')
print(prompt())
print(objects_to_answer(train[0]['objects']))
display(Image.open('report/figures/ground_truth_0.png'))


## 3. E0: zero-shot
E0 dùng model gốc NF4; output thô được lưu nguyên văn. Pipeline ở bước 7 chạy và chấm E0 trước huấn luyện.


In [ ]:
from solution.core import load_config
from solution.modeling import load_base, architecture_counts
from solution.experiment import infer, score_run
print(load_config('configs/e0.yaml'))


## 4. Dựng dữ liệu huấn luyện: chat template và mask loss
Chỉ answer/EOS có label; prefix và padding=-100. Hàm kiểm tra tiền tố chính xác trước khi train.


In [ ]:
from solution.modeling import encode_example, collate, encoding_audit
import inspect
print(inspect.getsource(encode_example))
print(inspect.getsource(collate))


## 5. Setup LoRA / đóng băng cho E1, E2, E3
LLM 252 targets; E3 thêm 96 ViT targets. E2/E3 lưu riêng projector full-tune.


In [ ]:
from solution.modeling import apply_setup
print(inspect.getsource(apply_setup))
for i in (1,2,3):
    print(load_config(f'configs/e{i}.yaml'))


## 6. Vòng huấn luyện
AdamW, warmup 5%, cosine, clipping 1.0, fp16/autocast, fp32 master weights; checkpoint 50 bước.


In [ ]:
from solution.training import train_sft
print(inspect.getsource(train_sft))
# Smoke test có tên riêng, không được dùng làm evidence nộp bài.
# subprocess.run([sys.executable, '-m', 'solution.experiment', 'train', '--config', 'configs/e1.yaml', '--steps', '2', '--work', str(WORK)], check=True)


## 7. Suy luận và chấm E0–E3; chạy E4 và E5
Cell này chạy toàn bộ pipeline và tiếp tục checkpoint nếu bị ngắt. Có thể cần nhiều giờ trên T4.
Chọn baseline bằng validation; không dùng test để chọn setup.


In [ ]:
subprocess.run([sys.executable, '-u', '-m', 'solution.experiment', 'suite', '--work', str(WORK)], check=True)


## 8. E4 (ablation dữ liệu)
10% và 100% của setup chọn bằng validation; giữ mọi biến khác. 100% tái sử dụng đúng baseline.


In [ ]:
from solution.storage import read_json
print(read_json('report/e4_design.json'))
print(read_json('report/evidence/e4_data_size_0.1.json'))
print(read_json('report/evidence/e4_data_size_1.0.json'))


## 9. Bonus E5 — GRPO
Reward có test: hoàn hảo, JSON lỗi, sai lớp, thừa box, rỗng, giant box. 4 generations, 128 tokens, 60 bước, use_vllm=False; so với SFT bằng số thật.


In [ ]:
from solution.core import iou_reward
print(inspect.getsource(iou_reward))
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'student_tests/test_core.py'], check=True)
print(read_json('report/evidence/e5.json'))


## 10. Báo cáo, kiểm tra và tải kết quả
Sinh báo cáo từ evidence; checker báo FAIL nếu chưa đủ kết quả hoặc chưa đạt ngưỡng rubric.


In [ ]:
subprocess.run([sys.executable, '-m', 'solution.reporting', '--work', str(WORK)], check=True)
import shutil
shutil.copy2('REPORT.md', WORK.parent/'REPORT.md')
subprocess.run([sys.executable, '-m', 'solution.check_quality'], check=True)
# Git commits kiểm tra trên repo local khi đưa kết quả về; runtime giải nén không có .git.
subprocess.run([sys.executable, 'scripts/check_submission.py'], check=True)
from google.colab import files
with zipfile.ZipFile('/content/day10_results.zip','w',zipfile.ZIP_DEFLATED) as z:
    z.write('REPORT.md')
    for p in Path('report').rglob('*'):
        if p.is_file(): z.write(p, p.as_posix())
files.download('/content/day10_results.zip')
display(Image.open('report/figures/learning_and_f1.png'))
